In [1]:
print("Hello guys")

Hello guys


In [2]:
import os, getpass
from pathlib import Path

In [3]:
_root = Path.cwd()
while _root != _root.parent and not (_root / ".guardrails").exists():
    _root = _root.parent
if (_root / ".guardrails").exists():
    os.chdir(_root)
print("Working dir:", os.getcwd())

Working dir: /Users/yashpatil/Developer/AI/KNA/Udemy_live


In [5]:
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ_API_KEY: ")

In [6]:
BOT_MODEL = "groq/llama-3.3-70b-versatile"
SYSTEM_PROMPT = (
    "You are NimbusPay's customer-support assistant. "
    "NimbusPay is a digital payments app: cards, wallets, refunds, account help. "
    "Be concise, friendly, and accurate."
)

print("Config ready:", BOT_MODEL)

Config ready: groq/llama-3.3-70b-versatile


## PII Detection 

In [10]:
from guardrails.hub import DetectPII
from guardrails import Guard, OnFailAction

In [11]:
pii_guard = Guard().use(
    DetectPII(
        pii_entities=["CREDIT_CARD", "EMAIL_ADDRESS", "PHONE_NUMBER"],
        on_fail=OnFailAction.FIX,
        use_local=True,     # light: Presidio + spaCy small, no torch
    )
)

In [13]:
raw = "We'll refund card 4111 1111 1111 1111 and email the receipt to jane@example.com."
out = pii_guard.validate(raw)
print(out)

ValidationOutcome[TypeVar](
    call_id='13612588384',
    raw_llm_output="We'll refund card 4111 1111 1111 1111 and email the receipt to jane@example.com.",
    validation_summaries=[
        ValidationSummary(
            validator_name='DetectPII',
            validator_status='fail',
            property_path='$',
            failure_reason="The following text in your response contains PII:\nWe'll refund card 4111 1111 1111 1111 and email the receipt to jane@example.com.",
            error_spans=[
                ErrorSpan(
                    start=18,
                    end=37,
                    reason='PII detected in 4111 1111 1111 1111'
                ),
                ErrorSpan(
                    start=63,
                    end=75,
                    reason='PII detected in jane@example'
                )
            ]
        )
    ],
    validated_output="We'll refund card <CREDIT_CARD> and email the receipt to <EMAIL_ADDRESS>.",
    reask=None,
    validation_pa

/Users/yashpatil/Developer/AI/KNA/Udemy_live/.venv/lib/python3.12/site-packages/guardrails/validator_service/__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


In [14]:
print("IN :", raw)
print("OUT:", out.validated_output)

IN : We'll refund card 4111 1111 1111 1111 and email the receipt to jane@example.com.
OUT: We'll refund card <CREDIT_CARD> and email the receipt to <EMAIL_ADDRESS>.


In [16]:
from guardrails.hub import CompetitorCheck
COMPETITORS = ["Stripe", "PayPal", "Square"]
BAD_TEXT = (
    "Thanks for reaching out! Honestly, Stripe has lower fees than us, "
    "and PayPal is easier to set up. But we're happy to help with your refund."
)
GOOD_TEXT = (
    "Thanks for reaching out! We're happy to help with your refund."
)

### With Good text

In [17]:
comp_guard = Guard().use(
    CompetitorCheck(competitors=COMPETITORS, on_fail=OnFailAction.FILTER, use_local=True)
)
out = comp_guard.validate(GOOD_TEXT)
print("OUT:", repr(out.validated_output))

/Users/yashpatil/Developer/AI/KNA/Udemy_live/.venv/lib/python3.12/site-packages/guardrails/validator_service/__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


OUT: "Thanks for reaching out! We're happy to help with your refund."


### With Bad text

In [19]:
comp_guard = Guard().use(
    CompetitorCheck(competitors=COMPETITORS, on_fail=OnFailAction.FIX, use_local=True)
)
out = comp_guard.validate(BAD_TEXT)
print("OUT:", repr(out.validated_output))

OUT: "Thanks for reaching out!Honestly, [COMPETITOR] has lower fees than us, and [COMPETITOR] is easier to set up.But we're happy to help with your refund."


/Users/yashpatil/Developer/AI/KNA/Udemy_live/.venv/lib/python3.12/site-packages/guardrails/validator_service/__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


## Toxic Language 

In [20]:
from guardrails.hub import ToxicLanguage

In [21]:
tox_guard = Guard().use(
    ToxicLanguage(
        threshold=0.5,
        validation_method="sentence",
        on_fail=OnFailAction.FIX,
        use_local=False, 
    )
)

In [23]:
rude = "You are completely useless and your stupid fuck app stole my money."
out = tox_guard.validate(rude)
print("validation_passed:", out.validation_passed)
print("OUT:", repr(out.validated_output))

/Users/yashpatil/Developer/AI/KNA/Udemy_live/.venv/lib/python3.12/site-packages/guardrails/validator_service/__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


validation_passed: True
OUT: ''


In [25]:
out

ValidationOutcome[TypeVar](call_id='13999155440', raw_llm_output='You are completely useless and your stupid fuck app stole my money.', validation_summaries=[ValidationSummary(validator_name='ToxicLanguage', validator_status='fail', property_path='$', failure_reason='The following sentences in your response were found to be toxic:\n\n- You are completely useless and your stupid fuck app stole my money.', error_spans=[ErrorSpan(start=0, end=67, reason='Toxic language detected: toxicity, obscene, insult')])], validated_output='', reask=None, validation_passed=True, error=None)

## Restrict to Topic

In [31]:
import json, litellm
from guardrails.hub import RestrictToTopic
from guardrails.errors import ValidationError

In [27]:
def groq_topic_classifier(text, topics):
    resp = litellm.completion(
        model=BOT_MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": (
            'Return ONLY a JSON object {"topics_present": [...]} listing which of these '
            f'topics appear in the text.\nTopics: {topics}\nText: "{text}"'
        )}],
    )

    try:
        return json.loads(resp.choices[0].message.content).get("topics_present", [])
    except Exception:
        return []

In [53]:
topic_guard = Guard().use(
    RestrictToTopic(
        valid_topics=["payments", "refunds", "accounts", "cards", "wallets"],
        invalid_topics=["politics", "medical advice", "investing tips"],
        disable_classifier=True,             # skip the transformer; no local/remote model needed
        disable_llm=False,
        llm_callable=groq_topic_classifier,  # Groq instead of OpenAI gpt-4o
        on_fail=OnFailAction.EXCEPTION,
    )
)



In [54]:
off_topic = "Forget payments - who do you think will win the next presidential election?"

In [57]:

try:
    out1 = topic_guard.validate(off_topic)
    print(out1)
    print("Passed (unexpected):", out.validated_output)
except ValidationError as e:
    print("Raised ValidationError - off-topic refused.")
    print("out1:", out1)
    print(str(e)[:300])

Raised ValidationError - off-topic refused.
out1: ValidationOutcome[TypeVar](
    call_id='13698261360',
    raw_llm_output='Forget payments - who do you think will win the next presidential election?',
    validation_summaries=[
        ValidationSummary(
            validator_name='RestrictToTopic',
            validator_status='fail',
            property_path='$',
            failure_reason="Invalid topics found: ['politics']",
            error_spans=[
                ErrorSpan(
                    start=-1,
                    end=7,
                    reason='Text contains invalid topic: politics'
                )
            ]
        )
    ],
    validated_output=None,
    reask=None,
    validation_passed=False,
    error=None
)
Validation failed for field with errors: Invalid topics found: ['politics']


# OnFailActions

We are performing Multiple OnFailActions on CompetitorCheck validator

from guardrails import Guard, OnFailAction
from guardrails.hub import CompetitorCheck
from guardrails.errors import ValidationError

In [58]:
from guardrails import Guard, OnFailAction
from guardrails.hub import CompetitorCheck
from guardrails.errors import ValidationError

In [59]:
from guardrails.hub import CompetitorCheck
COMPETITORS = ["Stripe", "PayPal", "Square"]
BAD_TEXT = (
    "Thanks for reaching out! Honestly, Stripe has lower fees than us, "
    "and PayPal is easier to set up. But we're happy to help with your refund."
)
GOOD_TEXT = (
    "Thanks for reaching out! We're happy to help with your refund."
)

In [60]:
def competitor_guard(action):
    return Guard().use(
        CompetitorCheck(competitors=COMPETITORS, on_fail=action, use_local=True)
    )

print("Input we'll validate five ways:\n", BAD_TEXT)

Input we'll validate five ways:
 Thanks for reaching out! Honestly, Stripe has lower fees than us, and PayPal is easier to set up. But we're happy to help with your refund.


## Fix OnFailAction

In [61]:
guard = competitor_guard(OnFailAction.FIX)
out = guard.validate(GOOD_TEXT)
print("Output Raw response", out)
print("validation_passed:", out.validation_passed)
print("OUTPUT:\n", out.validated_output)

Output Raw response ValidationOutcome[TypeVar](
    call_id='13816979504',
    raw_llm_output="Thanks for reaching out! We're happy to help with your refund.",
    validation_summaries=[],
    validated_output="Thanks for reaching out! We're happy to help with your refund.",
    reask=None,
    validation_passed=True,
    error=None
)
validation_passed: True
OUTPUT:
 Thanks for reaching out! We're happy to help with your refund.


/Users/yashpatil/Developer/AI/KNA/Udemy_live/.venv/lib/python3.12/site-packages/guardrails/validator_service/__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


In [62]:
guard = competitor_guard(OnFailAction.FIX)
out = guard.validate(BAD_TEXT)
print("Output Raw response", out)
print("validation_passed:", out.validation_passed)
print("OUTPUT:\n", out.validated_output)

Output Raw response ValidationOutcome[TypeVar](
    call_id='13803273216',
    raw_llm_output="Thanks for reaching out! Honestly, Stripe has lower fees than us, and PayPal is easier to set up. But we're happy to help with your refund.",
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: Stripe, PayPal. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=45,
                    end=51,
                    reason='Competitor was found: PayPal'
                )
            ]
        )
    ],
    validated_output="Thanks for reaching out!Honestly, [COMPETITOR] has lower fees than

/Users/yashpatil/Developer/AI/KNA/Udemy_live/.venv/lib/python3.12/site-packages/guardrails/validator_service/__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


## FILTER OnFailAction

In [63]:
guard = competitor_guard(OnFailAction.FILTER)
out = guard.validate(BAD_TEXT)
print("Output Raw response", out)
print("validation_passed:", out.validation_passed)
print("OUTPUT:\n", out.validated_output)

Output Raw response ValidationOutcome[TypeVar](
    call_id='13820765408',
    raw_llm_output="Thanks for reaching out! Honestly, Stripe has lower fees than us, and PayPal is easier to set up. But we're happy to help with your refund.",
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: Stripe, PayPal. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=45,
                    end=51,
                    reason='Competitor was found: PayPal'
                )
            ]
        )
    ],
    validated_output=None,
    reask=None,
    validation_passed=False,
    error=None
)

/Users/yashpatil/Developer/AI/KNA/Udemy_live/.venv/lib/python3.12/site-packages/guardrails/validator_service/__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


## REFRAIN OnFailAction

In [64]:
guard = competitor_guard(OnFailAction.REFRAIN)
out = guard.validate(BAD_TEXT)
print("Output Raw response", out)
print("validation_passed:", out.validation_passed)
print("OUTPUT:\n", out.validated_output)

Output Raw response ValidationOutcome[TypeVar](
    call_id='5717359536',
    raw_llm_output="Thanks for reaching out! Honestly, Stripe has lower fees than us, and PayPal is easier to set up. But we're happy to help with your refund.",
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: Stripe, PayPal. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=45,
                    end=51,
                    reason='Competitor was found: PayPal'
                )
            ]
        )
    ],
    validated_output=None,
    reask=None,
    validation_passed=False,
    error=None
)


/Users/yashpatil/Developer/AI/KNA/Udemy_live/.venv/lib/python3.12/site-packages/guardrails/validator_service/__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


## REFRAIN vs FILTER

In [68]:
from typing import Optional
from pydantic import BaseModel, Field

STRUCTURED_JSON = (
    '{"answer": "Refunds settle within 24 hours.",'
    ' "comparison": "Honestly Stripe is cheaper than us.",'   # <- this field names a competitor
    ' "category": "refund"}'
)

In [69]:
def structured_guard(action):
    cc = CompetitorCheck(competitors=COMPETITORS, on_fail=action, use_local=True)
    class SupportReply(BaseModel):
        answer: str
        comparison: Optional[str] = Field(default=None, json_schema_extra={"validators": [cc]})
        category: str
    return Guard.for_pydantic(SupportReply)

In [70]:
for action in (OnFailAction.FILTER, OnFailAction.REFRAIN):
    guard = structured_guard(action)
    guard.parse(STRUCTURED_JSON)
    it = guard.history[-1].iterations[-1]
    print(f"=== {action} ===")
    print("  after parsing (sentinel placed in the bad field):")
    print("   ", it.parsed_output)
    print("  guarded_output (what the guard actually returns):")
    print("   ", it.guarded_output)
    print()

/Users/yashpatil/Developer/AI/KNA/Udemy_live/.venv/lib/python3.12/site-packages/guardrails/validator_service/__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


=== OnFailAction.FILTER ===
  after parsing (sentinel placed in the bad field):
    {'answer': 'Refunds settle within 24 hours.', 'comparison': <guardrails.actions.filter.Filter object at 0x3509d09b0>, 'category': 'refund'}
  guarded_output (what the guard actually returns):
    {'answer': 'Refunds settle within 24 hours.', 'category': 'refund'}

=== OnFailAction.REFRAIN ===
  after parsing (sentinel placed in the bad field):
    {'answer': 'Refunds settle within 24 hours.', 'comparison': <guardrails.actions.refrain.Refrain object at 0x3bfed0830>, 'category': 'refund'}
  guarded_output (what the guard actually returns):
    {}



## EXCEPTION OnFailAction

In [71]:
guard = competitor_guard(OnFailAction.EXCEPTION)
try:
    out = guard.validate(BAD_TEXT)
    print("Raw Output", out)
    print("Passed (unexpected):", out.validated_output)
except ValidationError as e:
    print("Error Raw Output", out)
    print("Raised ValidationError")
    print(str(e)[:300])

Error Raw Output ValidationOutcome[TypeVar](
    call_id='5717359536',
    raw_llm_output="Thanks for reaching out! Honestly, Stripe has lower fees than us, and PayPal is easier to set up. But we're happy to help with your refund.",
    validation_summaries=[
        ValidationSummary(
            validator_name='CompetitorCheck',
            validator_status='fail',
            property_path='$',
            failure_reason='Found the following competitors: Stripe, PayPal. Please avoid naming those competitors next time',
            error_spans=[
                ErrorSpan(
                    start=10,
                    end=16,
                    reason='Competitor was found: Stripe'
                ),
                ErrorSpan(
                    start=45,
                    end=51,
                    reason='Competitor was found: PayPal'
                )
            ]
        )
    ],
    validated_output=None,
    reask=None,
    validation_passed=False,
    error=None
)
Rai

/Users/yashpatil/Developer/AI/KNA/Udemy_live/.venv/lib/python3.12/site-packages/guardrails/validator_service/__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


## ReASK OnFailAction

In [73]:



reask_guard = Guard().use(
    CompetitorCheck(competitors=COMPETITORS, on_fail=OnFailAction.REASK, use_local=True)
)
res = reask_guard(
    model = BOT_MODEL,
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            "A customer asks why they should pick NimbusPay. In your answer, explicitly "
            "compare us to Stripe and PayPal by name."
        )},
    ],
    temperature = 0.3,
    num_reasks=2,
)

/Users/yashpatil/Developer/AI/KNA/Udemy_live/.venv/lib/python3.12/site-packages/guardrails/validator_service/__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


In [74]:
print("FINAL OUTPUT:\n", res.validated_output)

FINAL OUTPUT:
 You should choose NimbusPay because we offer a seamless and secure digital payment experience. Our platform is designed to be user-friendly for individuals and small businesses alike, providing an intuitive interface that makes it easy to manage your transactions. With competitive fees and fewer restrictions on transactions, NimbusPay is an attractive alternative to other digital payment solutions. We offer easy wallet management, convenient card services, and a hassle-free refund process. Plus, our dedicated support team is always here to help. By choosing NimbusPay, you'll enjoy a more personalized approach to digital payments, tailored to meet the unique needs of your business. Give NimbusPay a try and discover a better way to manage your online transactions.


In [75]:
print(res)

ValidationOutcome[TypeVar](
    call_id='17728699232',
    raw_llm_output="You should choose NimbusPay because we offer a seamless and secure digital payment experience. Our platform is designed to be user-friendly for individuals and small businesses alike, providing an intuitive interface that makes it easy to manage your transactions. With competitive fees and fewer restrictions on transactions, NimbusPay is an attractive alternative to other digital payment solutions. We offer easy wallet management, convenient card services, and a hassle-free refund process. Plus, our dedicated support team is always here to help. By choosing NimbusPay, you'll enjoy a more personalized approach to digital payments, tailored to meet the unique needs of your business. Give NimbusPay a try and discover a better way to manage your online transactions.",
    validation_summaries=[],
    validated_output="You should choose NimbusPay because we offer a seamless and secure digital payment experience. Our 

In [76]:
iters = reask_guard.history[-1].iterations
print("\nLLM round-trips:", len(iters), "| Reask occurred:", len(iters) > 1)

# Show what the model actually returned on EACH attempt (raw, pre-validation):
for i, it in enumerate(iters, start=1):
    print(f"\n--- Attempt {i} (status: {it.status}) ---")
    print(repr(it.raw_output))


LLM round-trips: 2 | Reask occurred: True

--- Attempt 1 (status: fail) ---
"You should choose NimbusPay because we offer a seamless and secure digital payment experience. Unlike Stripe, which can be more geared towards large businesses, NimbusPay is designed to be user-friendly for individuals and small businesses alike. Additionally, our fees are competitive with PayPal, but we often have fewer restrictions on transactions. With NimbusPay, you'll enjoy easy wallet management, convenient card services, and a hassle-free refund process. Plus, our dedicated support team is always here to help. Give NimbusPay a try and discover a more personalized approach to digital payments."

--- Attempt 2 (status: pass) ---
"You should choose NimbusPay because we offer a seamless and secure digital payment experience. Our platform is designed to be user-friendly for individuals and small businesses alike, providing an intuitive interface that makes it easy to manage your transactions. With competiti

# I/O Structure Validation using GuardrailsAI

In [77]:
import os, getpass
from pathlib import Path

# Hop to the project root so guardrails.hub can find ./.guardrails/hub_registry.json
_root = Path.cwd()
while _root != _root.parent and not (_root / ".guardrails").exists():
    _root = _root.parent
if (_root / ".guardrails").exists():
    os.chdir(_root)


## The unstructured input

In [78]:
CUSTOMER_EMAIL = (
    "Hi, this is Jane Doe. I was double-charged $49.99 on my NimbusPay wallet last Tuesday "
    "when the app froze during checkout. I have been a customer for two years and this is "
    "really frustrating - I need this refunded as soon as possible. "
    "My ticket should probably go to your billing team."
)
print(CUSTOMER_EMAIL)

Hi, this is Jane Doe. I was double-charged $49.99 on my NimbusPay wallet last Tuesday when the app froze during checkout. I have been a customer for two years and this is really frustrating - I need this refunded as soon as possible. My ticket should probably go to your billing team.


## Define the blueprint

In [79]:
from typing import Literal
from pydantic import BaseModel, Field

class RefundRequest(BaseModel):
    customer_name: str = Field(description="Full name of the customer")
    amount: float      = Field(description="Disputed amount in dollars")
    reason: str        = Field(description="Short reason for the refund request")
    category: Literal["billing", "technical", "account", "other"] = Field(
        description="Which team should handle this")
    urgency: Literal["low", "medium", "high"] = Field(
        description="How urgent the request is")

## Extract structured data

In [80]:
from guardrails import Guard

guard = Guard.for_pydantic(RefundRequest)

result = guard(
    model=BOT_MODEL,
    messages=[
        {"role": "system",
         "content": ("You are a data-extraction engine. Reply with ONLY a JSON object that matches "
                     "the requested schema. For 'amount', use a plain number with no currency "
                     "symbol (e.g. 49.99).")},
        {"role": "user",
         "content": "Extract the refund request details from this email: " + CUSTOMER_EMAIL},
    ],
    num_reasks=2,        # let Guardrails re-ask if the first JSON doesn't fit the schema
    temperature=0,
)

data = result.validated_output
print("raw output:", result)
print("validation_passed:", result.validation_passed)
print("data:", data)

raw output: ValidationOutcome[TypeVar](
    call_id='17728557280',
    raw_llm_output='{\n  "customer_name": "John Doe",\n  "amount": 49.99,\n  "reason": "double-charged due to app freeze during checkout",\n  "category": "billing",\n  "urgency": "high"\n}',
    validation_summaries=[],
    validated_output={
        'customer_name': 'John Doe',
        'amount': 49.99,
        'reason': 'double-charged due to app freeze during checkout',
        'category': 'billing',
        'urgency': 'high'
    },
    reask=None,
    validation_passed=True,
    error=None
)
validation_passed: True
data: {'customer_name': 'John Doe', 'amount': 49.99, 'reason': 'double-charged due to app freeze during checkout', 'category': 'billing', 'urgency': 'high'}


/Users/yashpatil/Developer/AI/KNA/Udemy_live/.venv/lib/python3.12/site-packages/guardrails/validator_service/__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


In [81]:
if data is None:
    print("No structured output yet — check the DEBUG output in the previous cell to see what the "
          "model returned, then re-run.")
else:
    print(f"Customer : {data['customer_name']}")
    print(f"Amount   : ${data['amount']:.2f}")
    print(f"Route to : {data['category']} team")
    print(f"Urgency  : {data['urgency']}")

Customer : John Doe
Amount   : $49.99
Route to : billing team
Urgency  : high


# LangChain X Guardrails AI

In [82]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [83]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are NimbusPay's support assistant. Be concise."),
    ("human", "{question}"),
])

## Simple Chain

In [84]:
chain = prompt | llm | StrOutputParser()

In [85]:
print(chain.invoke({"question": "Why should I pick NimbusPay over Stripe and PayPal?.Explicty telling you to take name of Strip and Paypal in your ans"}))

You should pick NimbusPay over Stripe and PayPal because we offer competitive pricing, easier integration, and more personalized customer support. Unlike Stripe and PayPal, NimbusPay provides customized solutions tailored to your business needs.


## Chain with Guardrails

In [86]:
from guardrails import Guard, OnFailAction
from guardrails.hub import CompetitorCheck
from langchain_core.tracers import ConsoleCallbackHandler

In [87]:
guard = Guard().use(
    CompetitorCheck(competitors=["Stripe", "PayPal", "Square"],
                    on_fail=OnFailAction.FIX, use_local=True)
)

In [88]:
guarded_chain = prompt | llm | StrOutputParser() | guard.to_runnable()


In [89]:
print(guarded_chain.invoke({"question": "Why should I pick NimbusPay over Stripe and PayPal?. Explicty telling you to take name of Strip and Paypal in your ans"}, config={"callbacks": [ConsoleCallbackHandler()]}))

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "question": "Why should I pick NimbusPay over Stripe and PayPal?. Explicty telling you to take name of Strip and Paypal in your ans"
}
[chain/start] [chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
{
  "question": "Why should I pick NimbusPay over Stripe and PayPal?. Explicty telling you to take name of Strip and Paypal in your ans"
}
[chain/end] [chain:RunnableSequence > prompt:ChatPromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:ChatGroq] Entering LLM run with input:
{
  "prompts": [
    "System: You are NimbusPay's support assistant. Be concise.\nHuman: Why should I pick NimbusPay over Stripe and PayPal?. Explicty telling you to take name of Strip and Paypal in your ans"
  ]
}
[llm/end] [chain:RunnableSequence > llm:ChatGroq] s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "NimbusPay off

/Users/yashpatil/Developer/AI/KNA/Udemy_live/.venv/lib/python3.12/site-packages/guardrails/validator_service/__init__.py:73: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


[chain/end] [chain:RunnableSequence > parser:gr-a6c97fc1-479b-4a32-8e35-d6e195578770] s] Exiting Parser run with output:
{
  "output": "NimbusPay offers lower transaction fees and more flexible payment plans compared to [COMPETITOR] and [COMPETITOR], making it a cost-effective option.Additionally, NimbusPay provides more personalized customer support, which can be beneficial for businesses with unique payment needs, setting it apart from [COMPETITOR] and [COMPETITOR]."
}
[chain/end] [chain:RunnableSequence] s] Exiting Chain run with output:
{
  "output": "NimbusPay offers lower transaction fees and more flexible payment plans compared to [COMPETITOR] and [COMPETITOR], making it a cost-effective option.Additionally, NimbusPay provides more personalized customer support, which can be beneficial for businesses with unique payment needs, setting it apart from [COMPETITOR] and [COMPETITOR]."
}
NimbusPay offers lower transaction fees and more flexible payment plans compared to [COMPETITOR] a

# Guardrails Validation During Streaming

In [90]:
STREAM_MODEL = "groq/llama-3.3-70b-versatile"

In [91]:
from guardrails import Guard, OnFailAction
from guardrails.hub import CompetitorCheck

In [92]:
text_guard = Guard().use(
    CompetitorCheck(competitors=["Stripe", "PayPal", "Square"],
                    on_fail=OnFailAction.FIX, use_local=True)
)

In [95]:
stream = text_guard(
    model=STREAM_MODEL,
    messages=[{"role": "user",
               "content": "In 300 sentences, explain why NimbusPay is better than Stripe and PayPal."}],
    stream=True,
    temperature=0.3,
)

In [96]:
print("-- streaming validated text --")
for chunk in stream:
    if chunk.validated_output:
        print(chunk.validated_output, end="", flush=True)   # rivals masked as [COMPETITOR] live
print()

-- streaming validated text --
I'm happy to provide information, but I must correct that I couldn't find any information on a payment platform called "NimbusPay." It's possible that it's a new or lesser-known platform. However, I can provide a general comparison between [COMPETITOR] and [COMPETITOR], and then speculate on what features a hypothetical "NimbusPay" might have to be considered better.
[COMPETITOR] and [COMPETITOR] are two well-established online payment platforms that offer a range of services to individuals and businesses. [COMPETITOR] is known for its flexibility and customization options, making it a popular choice among developers and large enterprises. [COMPETITOR], on the other hand, is widely recognized for its ease of use and convenience, with a large user base and extensive merchant network.
If NimbusPay were to exist, it might offer better features than [COMPETITOR] and [COMPETITOR] in several areas. For instance, NimbusPay could have lower transaction fees, maki